In [1]:
import os
from google.cloud import bigquery
from dotenv import load_dotenv
import vertexai
from google.cloud import discoveryengine_v1 as discoveryengine
from vertexai import generative_models as genai  # Añadir esta línea
from vertexai.generative_models import (
    FunctionDeclaration,
    GenerationConfig,
    Tool,
)

In [2]:
load_dotenv()  # Carga las variables desde .env al entorno
client = bigquery.Client(project='dataton-2024-team-01-cofares')
# Ahora puedes acceder a las variables de entorno
project_id = os.getenv("GOOGLE_CLOUD_PROJECT")

In [3]:
# Configuración del cliente de Vertex AI
PROJECT_ID = "dataton-2024-team-01-cofares"
LOCATION = "us-central1"
vertexai.init(project=PROJECT_ID, location=LOCATION)
#multimodal_model = GenerativeModel("gemini-1.5-flash-001")

# Inicializa el cliente de Discovery Engine
discovery_client = discoveryengine.RankServiceClient()  

In [4]:

def get_products(prompt):
    client = bigquery.Client(project=project_id)
    query = """
    WITH QueryEmbedding AS (
      SELECT
        ml_generate_embedding_result AS query_embedding
      FROM
        ML.GENERATE_EMBEDDING(
          MODEL `dataton-2024-team-01-cofares.datos_cofares.text_embedding`,
          (SELECT @prompt AS content),  -- Aquí usamos el parámetro
          STRUCT(TRUE AS flatten_json_output, 'RETRIEVAL_QUERY' AS task_type)
        )
    )
    SELECT
      d.nombre_completo_material AS nombre,
      d.txt_mas_informacion_del_producto AS descripcion,
      d.txt_instrucciones_de_uso AS modo_implementacion,
      d.codigo_web,
      d.URI_primera_imagen,
      d.codigo_nacional,
      ML.DISTANCE(
        qe.query_embedding,
        e.ml_generate_embedding_result,
        'COSINE'
      ) AS distance_to_query
    FROM
      `dataton-2024-team-01-cofares.datos_cofares.data_final_temp` AS d
    INNER JOIN
      `dataton-2024-team-01-cofares.datos_cofares.SalidaEmbeddings_temp` AS e
      ON d.codigo_web = e.title
    INNER JOIN QueryEmbedding AS qe
      ON TRUE
    ORDER BY
      distance_to_query
    LIMIT 10;
    """.format(prompt)
    # Configura el parámetro para el prompt
    job_config = bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter("prompt", "STRING", prompt)
        ]
    )

    query_job = client.query(query, job_config=job_config)
    results = query_job.result()
    
    products = []
    for row in results:

        descripcion = row.descripcion
        if not row.descripcion:
            descripcion = '-'
        
        modo_implementacion = row.modo_implementacion
        if not row.modo_implementacion:
            modo_implementacion = '-'


        # Cambia la URL si es necesario
        imagen_url = row.URI_primera_imagen 
        if imagen_url and imagen_url.startswith('gs:/'):
            imagen_url = imagen_url.replace('gs://dataton-2024-team-01-cofares-datastore/imagenes/', 'https://storage.googleapis.com/dataton-2024-team-01-cofares-datastore/imagenes/reto_cofares/')
        products.append({
            "codigo_web": row.codigo_web,
            "nombre": row.nombre,
            "codigo_nacional": row.codigo_nacional,
            "descripcion": descripcion,
            "modo_implementacion": modo_implementacion,
            "imagen_url": imagen_url,
            "distance_to_query": row.distance_to_query
        })
    return products

In [5]:
def rerank_products(prompt, products):
    ranking_config = discovery_client.ranking_config_path(
        project=PROJECT_ID,
        location=LOCATION,
        ranking_config="default_ranking_config",
    )
    
    records = [
        discoveryengine.RankingRecord(
            id=str(index),
            title=product["nombre"],
            content=product["descripcion"] + " " + product["modo_implementacion"]
        )
        for index, product in enumerate(products)
    ]
    
    request = discoveryengine.RankRequest(
        ranking_config=ranking_config,
        model="semantic-ranker-512@latest",
        top_n=10, # cantidad de productos a rankear
        query=prompt,
        records=records,
    )
    
    response = discovery_client.rank(request=request)
    
    # Aseguramos que los productos están formateados según el esquema
    ranked_products = [
        {
            "codigo_web": products[int(record.id)]["codigo_web"],
            "nombre": products[int(record.id)]["nombre"],
            "codigo_nacional": products[int(record.id)]["codigo_nacional"],
            "descripcion": products[int(record.id)]["descripcion"],
            "modo_implementacion": products[int(record.id)]["modo_implementacion"],
            "imagen_url": products[int(record.id)]["imagen_url"],
            "distance_to_query": products[int(record.id)]["distance_to_query"]
        }
        for record in response.records[:5] # cantidad de productos a mostrar
    ]
    
    return {"products": ranked_products}

In [6]:
#FUNCTION CALLING

# Define el schema
product_schema = FunctionDeclaration(
    name="product_query",
    description="Fetches relevant product information based on a search prompt.",
    parameters={
        "type": "object",
        "properties": {
            "products": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "codigo_web": {"type": "string", "description": "Product web code"},
                        "nombre": {"type": "string", "description": "Product name"},
                        "codigo_nacional": {"type": "string", "description": "National product code"},
                        "descripcion": {"type": "string", "description": "Product description"},
                        "modo_implementacion": {"type": "string", "description": "Mode of implementation"},
                        "imagen_url": {"type": "string", "description": "Image URL"},
                        "distance_to_query": {"type": "number", "description": "Semantic distance to query"}
                    }
                }
            }
        }
    }
)

# Define tools antes de inicializar el modelo
tools = [Tool(function_declarations=[product_schema])]

In [7]:
PROJECT_ID = "dataton-2024-team-01-cofares"  # @param {type:"string"}
LOCATION = "us-central1"  # @param {type:"string"}

# Importa el modelo de Gemini Flash 1.5
import vertexai
vertexai.init(project=PROJECT_ID, location=LOCATION)


# Model definition
multimodal_model = genai.GenerativeModel(
"gemini-1.5-flash",
generation_config=GenerationConfig(temperature=0),
tools=tools)

chat = multimodal_model.start_chat(response_validation=False)

In [8]:
def generate_response(prompt):  # Eliminamos el parámetro products
    #chat = multimodal_model.start_chat()

    instruction_prompt = f"""
    Eres un asistente farmacéutico experto. 
    
    Consulta recibida de un farmacéutico: "{prompt}"
    
    Por favor, responde de la siguiente manera:
    
    - Saluda a los usuarios y pregúntales en qué puedes ayudarles hoy.
    - Resume la petición del usuario y pídale que confirme que ha entendido correctamente.
    - Si es necesario, pida detalles aclaratorios.
    - Utilice ${tools} para recibir un listado de productos rankeados para ayudar al usuario con su tarea.
    - Agradezca al usuario su colaboración y despídase.
    """

    try:
        response = chat.send_message(instruction_prompt)
        response.candidates[0].content.parts[0]
        
        # Verificar si hay una llamada a función
        for candidate in response.candidates:
            for part in candidate.content.parts:
                if hasattr(part, 'function_call') and part.function_call:
                    # Ejecutar búsqueda de productos
                    products = get_products(prompt)
                    if not products:
                        return "Lo siento, no encontré productos que coincidan con tu búsqueda."
                    
                    ranked_products = rerank_products(prompt, products)
                    
                    # Enviar los resultados al modelo para generar una respuesta contextual
                    results_prompt = f"""
                    Basado en la búsqueda "{prompt}", he encontrado estos productos:
                    {[product['nombre'] for product in ranked_products['products']]}
                    
                    Por favor, genera una respuesta útil que:
                    1. Mencione los productos encontrados
                    2. Explique por qué son relevantes
                    3. Proporcione recomendaciones de uso
                    """
                    
                    final_response = chat.send_message(results_prompt)
                    return {
                        "type": "product_search",
                        "message": final_response.text,
                        "products": ranked_products["products"]
                    }
                
        # Si no hay llamada a función, devolver la respuesta conversacional
        return {
            "type": "conversation",
            "message": response.text
        }
                    
    except Exception as e:
        return f"Lo siento, ocurrió un error: {str(e)}"

In [25]:
# Ejemplo de uso
prompt = "¿Hay algún spray nasal para alergias?"

In [26]:

products = get_products(prompt)  # Llamar a la función para obtener productos
# Imprimir los productos obtenidos
print("Productos obtenidos:")
for product in products:
    print(f"Nombre: {product['nombre']}, Descripción: {product['descripcion']}, Modo de implementación: {product['modo_implementacion']}, Distancia: {product['distance_to_query']}")

Productos obtenidos:
Nombre: Pack Bexident Fresh Breath Colutorio + Spray, Descripción: La gama Fresh Breath de Bexident ayuda a combatir el mal aliento neutralizándolo y combatiendo los sulfuros volátiles que lo causan. Además, alivia la sequedad bucal y tiene propiedades antisépticas., Modo de implementación: Se recomienda usar el colutorio 3 veces al día, después del cepillado y durante, al menos, 30 segundos. Se recomienda pulverizar el spray 3 o 4 veces en la boca y no enjuagar.IndicacionesIndicado para conseguir una buena higiene bucal y un aliento fresco y sano.ContraindicacionesEvitar el contacto con los ojos. No ingerir. Mantener fuera del alcance de los niños., Distancia: 0.44051285926191364
Nombre: FARLINE DUPLO FRIMAR BABY AGUA DE MAR + GASA BEBÉ REGALO 2 x 120 MILILITROS, Descripción: -, Modo de implementación: -, Distancia: 0.4449157475821408
Nombre: Bach Rescue Spray, 20 ml, Descripción: Este spray está formulado a base de CherryPlum, Clematis, Impatiens, Rock Rose y Sta

In [27]:
# Llamar a la función de reranking
ranked_products = rerank_products(prompt, products)["products"]  # Accede a la lista de productos

# Imprimir los productos rankeados
print("Productos rankeados:")
for product in ranked_products:
    print(f"Nombre: {product['nombre']}, Descripción: {product['descripcion']}, Modo de implementación: {product['modo_implementacion']}, Distancia: {product['distance_to_query']}")

Productos rankeados:
Nombre: Duplo Farline Desodorante Spray Sensible, 2 Uds, Descripción: Farline Desodorante Spray Sensible no contiene sales de aluminio y aporta hasta 24 horas de protección. Apto para pieles sensibles.Sin alcohol.Testado dermatologicamente., Modo de implementación: Te recomendamos aplicar Duplo Farline Desodorante Spray Sensible manteniendo el envase en posición vertical y vaporizar a una distancia de 15 cm de la axila.IndicacionesIndicado para adultos.ContraindicacionesMantener fuera del alcance de los niños.Conservar en un lugar fresco y seco.Evitar el contacto con los ojos y la boca.No ingerir., Distancia: 0.47837449336449844
Nombre: Duplo Farline Desodorante Spray Extra_Dry, 2 Uds, Descripción: Farline Desodorante Spray Extra-Dry ayuda a controlar la sudoración excesiva, absorbiendo el exceso de humedad y aportando una máxima protección.Sin alcohol.Testado bajo control dermatológico., Modo de implementación: Te recomendamos aplicar Farline Desodorante Spray Ext

In [28]:
response_text = generate_response(prompt)  # Generar la respuesta
print(response_text)

{'type': 'product_search', 'message': '¡Hola! He encontrado algunos productos que podrían ser útiles para las alergias, pero ninguno es un spray nasal específico:\n\n* **Bach Rescue Spray, 20 ml:** Este spray contiene una mezcla de flores de Bach, que se utilizan para aliviar el estrés y la ansiedad. Puede ser útil para aliviar los síntomas de alergia, como la ansiedad y la irritabilidad. Se recomienda aplicar unas gotas debajo de la lengua o en un vaso de agua, según las indicaciones del fabricante.\n\nLos otros productos que mencionaste no son relevantes para las alergias:\n\n* **Duplo Farline Desodorante Spray Sensible, 2 Uds:** Este producto es un desodorante para axilas.\n* **Duplo Farline Desodorante Spray Extra_Dry, 2 Uds:** Este producto es un desodorante para axilas.\n* **Duplo Farline Desodorante para Hombre en Spray, 2 x 150 ml:** Este producto es un desodorante para axilas.\n* **Pack Bexident Fresh Breath Colutorio + Spray:** Este pack contiene un colutorio y un spray para 